# ClashBot: Current-Deck BC + RL Training on Google Colab
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/varun-zero/clash-ml-model/blob/main/ClashBot_Colab_Training.ipynb)

Trains the deployed agent on the deck the account actually plays:

**Knight - Mini PEKKA - Archers - Musketeer - Arrows - Fireball - Giant - Minions**

### Pipeline
1. **Behaviour cloning** on the tactical teacher (`TacticalCoachBot`): defence-first, spell-disciplined, placement-varied.
2. **PPO/GAE reinforcement learning** against scripted bots with the same deck.
3. **Public-information opponent elixir** via `OpponentElixirTracker` (no hidden-state leak).
4. Google Drive persistence for checkpoints, history and curves.

### What changed vs. the old pipeline
| Problem in live play | Fix |
|---|---|
| Fireball thrown at our own tower / enemy king tower (activating it) | `tactics.evaluate_spell_cast` blocks own-half and king-tower casts, rewards value casts |
| Kept pushing while our towers were being destroyed | defence-priority shaping: pushing while threatened is penalised, defending is rewarded |
| Always deployed on the same 3-4 tiles | placement anchors + jitter in the teacher, placement-variety shaping, tile entropy in PPO |
| Opponent elixir read from the simulator's private counter | explicit `OpponentElixirTracker` built from public plays only |

## 1. Verify Tesla T4 GPU & CUDA Acceleration

In [ ]:
!nvidia-smi
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")

## 2. Mount Google Drive for Continuous Persistence

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/clashbot_training'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f"Google Drive Mounted! Checkpoint sync path: {DRIVE_DIR}")

## 3. Clone Repository & Install Dependencies

In [ ]:
!git clone https://github.com/varun-zero/clash-ml-model.git /content/clashbot
%cd /content/clashbot
!pip install -q -r requirements.txt

import os, sys
cwd = os.getcwd()
if cwd not in sys.path:
    sys.path.insert(0, cwd)
print(f"Working Directory: {cwd}")

## 4. Confirm the Deck & Sanity-Check the Trainer

In [ ]:
from cr_alphastar_agent.config import CURRENT_DECK, CURRENT_DECK_IDS, CARDS_CATALOG, resolve_deck_ids
from cr_alphastar_agent.env.opponent_elixir import OpponentElixirTracker

print("Deck:", CURRENT_DECK)
print("Policy IDs:", CURRENT_DECK_IDS, "->", [CARDS_CATALOG[c]["name"] for c in CURRENT_DECK_IDS])
assert resolve_deck_ids('current') == CURRENT_DECK_IDS
assert len(set(CURRENT_DECK_IDS)) == 8

# Opponent elixir tracker sanity check (public information only).
t = OpponentElixirTracker()
t.observe_play(cost=4, t_s=30.0)
print("Tracker estimate:", t.estimate(30.0).to_dict())

## 5. Behaviour Cloning on the Current Deck

Clones `TacticalCoachBot` (defence-first, spell-disciplined, placement-varied) and writes `checkpoints/bc.pt`.

Useful knobs: `--num-matches` (more = more diverse data), `--teacher coach|tactical|scripted`, `--epochs`.

In [ ]:
!python warmstart_bc.py \
    --deck current \
    --teacher coach \
    --num-matches 60 \
    --epochs 4 \
    --lr 1e-3 \
    --checkpoint-dir ./checkpoints \
    --out-name bc.pt

## 6. Reinforcement Learning (PPO + GAE) vs Scripted Bots

The learner keeps the current deck; the opponent draws varied decks so the policy does not overfit a single matchup.

`--opponents scripted` trains against the rule bots first. Switch to `mixed` (ramps into the PFSP league) or `self_play` for league-only training.

In [ ]:
!python train_colab.py \
    --total-updates 1000 \
    --num-envs 8 \
    --rollout-steps 256 \
    --deck current \
    --opponents scripted \
    --init-bc ./checkpoints/bc.pt \
    --checkpoint-dir ./checkpoints \
    --drive-dir /content/drive/MyDrive/clashbot_training

## 7. Continue Against the League (Optional)

After the scripted-bot stage, continue with the PFSP league (historical snapshots + mirror self-play) on the same deck.

In [ ]:
!python train_colab.py \
    --total-updates 500 \
    --num-envs 8 \
    --rollout-steps 256 \
    --deck current \
    --opponents self_play \
    --checkpoint-dir ./checkpoints \
    --drive-dir /content/drive/MyDrive/clashbot_training

## 8. View Real-Time Improvement & Telemetry Curves

In [ ]:
from IPython.display import Image, display
import os, json

plot_path = "checkpoints/clashbot_training_curve.png"
if not os.path.exists(plot_path):
    plot_path = "/content/drive/MyDrive/clashbot_training/clashbot_training_curve.png"
if os.path.exists(plot_path):
    display(Image(plot_path))
else:
    print("Training plot will be generated after the first PPO batch!")

hist_path = "checkpoints/training_history.json"
if os.path.exists(hist_path):
    hist = json.load(open(hist_path))
    if hist:
        last = hist[-1]
        print(f"step={last['step']} update={last['update']} win_rate={last['win_rate']:.1f}% ")
        print(f"record: {last['total_wins']}W-{last['total_losses']}L-{last['total_draws']}D over {last['total_matches']} matches")

## 9. Download the Final Checkpoint

`latest.pt` is the deployable checkpoint (same format as `deploy_bluestacks.py` expects).

In [ ]:
import shutil, os
if os.path.exists("checkpoints/latest.pt"):
    shutil.copy2("checkpoints/latest.pt", "/content/drive/MyDrive/clashbot_training/latest.pt")
    print("Saved latest.pt to Google Drive")
else:
    print("No latest.pt yet - run the training cell first.")